In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas11") \
    .master("local[*]") \
    .getOrCreate()

print("Versi Spark:", spark.version)

26/11/17 22:22:19 WARN Utils: Your hostname, cahyoalim-ThinkPad-L14-Gen-2 resolves to a loopback address: 127.0.1.1; using 192.168.1.194 instead (on interface wlp9s0)
26/11/17 22:22:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 22:22:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Versi Spark: 3.5.9


In [2]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [9]:
print("SOAL A: Persiapan Data & Feature Engineering\n")

from pyspark.ml.feature import StringIndexer, VectorAssembler

# Membaca file CSV dataset yang sudah dibuat sebelumnya
tugas = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
print("Jumlah baris data:", tugas.count())
tugas.show(5)

# Merubah kolom teks menjadi numerik 
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

#Menerapkan indexer ke dataframe
model_indexer_menu = indexer_menu.fit(tugas)
tugas = model_indexer_menu.transform(tugas)

model_indexer_cabang = indexer_cabang.fit(tugas)
tugas = model_indexer_cabang.transform(tugas)

model_indexer_waktu = indexer_waktu.fit(tugas)
tugas = model_indexer_waktu.transform(tugas)

# Menggabungkan beberapa kolom menjadi satu kolom vector
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

tugas_final = assembler.transform(tugas)

SOAL A: Persiapan Data & Feature Engineering

Jumlah baris data: 2500
+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [4]:
print("SOAL B")

from pyspark.ml.regression import LinearRegression

#Membagi data untuk training dan testing
train, test = tugas_final.randomSplit([0.75, 0.25], seed=101)

#Mendefinisikan dan melatih model
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

SOAL B


26/11/17 22:24:21 WARN Instrumentation: [88d492b0] regParam is zero, which might cause numerical instability and overfitting.


Koefisien tiap fitur: [22046.663271025755,3.0229208681606696,-123.21425598657821,279.28452461676096,339.63879532317225]
Intercept: -66903.1776648588


In [6]:
print("SOAL C\n")

from pyspark.ml.evaluation import RegressionEvaluator

#prediksi pada testing
predictions = model.transform(test)

predictions.select("qty", "harga_satuan", "total_penjualan", "prediction").show()

#Menghitung RMSE 
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse_score = evaluator_rmse.evaluate(predictions)

#Menghitung metrik R²
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2_score = evaluator_r2.evaluate(predictions)

print("\nhasil evaluasi model")
print(f"Nilai RMSE : {rmse_score:,.2f}")
print(f"Nilai R²   : {r2_score:.4f}")


SOAL C

+---+------------+------------------+------------------+
|qty|harga_satuan|   total_penjualan|        prediction|
+---+------------+------------------+------------------+
|  1|       25000|19721.170446051627|31056.146105506865|
|  1|       25000|  21236.0956379149|31056.146105506865|
|  4|       22000|  85834.8564401191| 88127.37331410211|
|  4|       22000|102174.09100317833| 88127.37331410211|
|  4|       30000|113044.63431555482|112310.74025938749|
|  5|       22000|115810.29600214415|110174.03658512786|
|  5|       25000|140386.40273615732|119242.79918960987|
|  2|       30000| 73842.64420790318|  67877.7749220128|
|  3|       15000|32345.642983082646| 44580.62517062851|
|  4|       15000| 45510.16712269087| 66627.28844165425|
|  4|       20000| 85362.59400453049|  81741.8927824576|
|  4|       22000|  95663.3347901574| 87787.73451877893|
|  4|       22000| 97390.04610730882| 87787.73451877893|
|  4|       25000| 99994.17567791023| 96856.49712326094|
|  4|       30000|13075

In [12]:
print("SOAL D\n")

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

#menggabungkan kolom yang ditentukan
assembler_dua = VectorAssembler(
    inputCols=["qty", "harga_satuan"], 
    outputCol="features_dua"
)

tugas_eksperimen = assembler_dua.transform(tugas)

#Membagi data untuk pelatihan dan pengujian
train_dua, test_dua = tugas_eksperimen.randomSplit([0.75, 0.25], seed=101)

#melatih model kedua
lr_dua = LinearRegression(featuresCol="features_dua", labelCol="total_penjualan")
model_dua = lr_dua.fit(train_dua)

#menguji model kedua dana menghitung matirk R^2
predictions_dua = model_dua.transform(test_dua)
evaluator_r2_dua = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2_score_dua = evaluator_r2_dua.evaluate(predictions_dua)

#Menampilkan perbandingan
print(f"Nilai R² Model Utama (5 fitur) : {r2_score:.4f}")
print(f"Nilai R² Model Kedua (2 fitur) : {r2_score_dua:.4f}")

SOAL D



26/11/17 23:47:50 WARN Instrumentation: [d31c35da] regParam is zero, which might cause numerical instability and overfitting.


Nilai R² Model Utama (5 fitur) : 0.9107
Nilai R² Model Kedua (2 fitur) : 0.9108


In [11]:
import pandas as pd

print("SOAL E\n")

#Membuat 3 data dummy 
data_skenario = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Roti Bakar", "Americano"],
    "cabang": ["Semarang", "Yogyakarta", "Magelang"],
    "waktu_pesan": ["Malam", "Pagi", "Siang"],
    "qty": [2, 1, 5],
    "harga_satuan": [15000, 20000, 18000]
})

#Konversi ke Spark DataFrame
df_skenario = spark.createDataFrame(data_skenario)
df_skenario.show()

#Mengubah teks ke indeks menggunakan model StringIndexer DARI SOAL A
df_skenario = model_indexer_menu.transform(df_skenario)
df_skenario = model_indexer_cabang.transform(df_skenario)
df_skenario = model_indexer_waktu.transform(df_skenario)

#Menggabungkan fitur menggunakan VectorAssembler DARI SOAL A
df_skenario_final = assembler.transform(df_skenario)

#Melakukan prediksi menggunakan model UTAMA dari Soal B
hasil_prediksi = model.transform(df_skenario_final)

hasil_prediksi.select("nama_menu", "qty", "harga_satuan", "prediction").show()

SOAL E

+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Kopi Susu|  Semarang|      Malam|  2|       15000|
|Roti Bakar|Yogyakarta|       Pagi|  1|       20000|
| Americano|  Magelang|      Siang|  5|       18000|
+----------+----------+-----------+---+------------+

+----------+---+------------+------------------+
| nama_menu|qty|harga_satuan|        prediction|
+----------+---+------------+------------------+
| Kopi Susu|  2|       15000|23308.955488172884|
|Roti Bakar|  1|       20000|15388.330470050787|
| Americano|  5|       18000| 98421.99190780836|
+----------+---+------------+------------------+



In [6]:
spark.stop()